# Gerador de Cartas de Magic: The Gathering

Projeto da disciplina de IA e Bibliotecas de Dados para Python.

**Como usar:** no menu acima, clique em **Ambiente de execução > Executar tudo** (ou `Ctrl+F9`). As duas primeiras células preparam o ambiente (clonam o repositório e instalam as dependências); a terceira mostra o formulário para gerar a carta.

O código completo do pipeline de geração está em [`src/pipeline/generator.py`](../src/pipeline/generator.py) no repositório — este notebook só importa e chama a função `gerar_carta`, para que a mesma lógica possa também ser usada por outras interfaces no futuro (ex: um app web).

## 1. Preparar o ambiente

Clona o repositório (se ainda não estiver presente) e instala as dependências.

In [ ]:
import os

REPO_URL = "https://github.com/funkymonks21/MTGCardCreator-IFRS.git"
REPO_DIR = "MTGCardCreator-IFRS"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL}
else:
    # a pasta ja existe de uma execucao anterior nesta sessao do Colab --
    # puxa os commits novos em vez de ignorar
    !cd {REPO_DIR} && git pull -q

%cd {REPO_DIR}
!pip install -q -r requirements.txt

import sys
sys.path.insert(0, "src")

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt_tab", quiet=True)
# POS tagging usado pela inferencia automatica de tipo (ver
# src/pipeline/tipo_por_nome.py). Os dois nomes cobrem versoes
# diferentes do NLTK.
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

print("Ambiente pronto.")


## 2. Carregar o pipeline

Importa a função `gerar_carta`. Dois conjuntos de modelos são carregados automaticamente, sob demanda, direto do repositório, então ninguém precisa treinar nada aqui:

- **texto** -- cadeia de Markov treinada por tipo e cor (`models/modelo_markov.pkl`, ver `src/pipeline/texto_markov.py`);
- **custo de mana e poder/resistência** -- regressões scikit-learn (`models/modelos_estruturais.pkl`, ver `src/pipeline/estrutura.py`).

O pipeline decide primeiro o **tipo** (Criatura, Instantânea, Feitiço, Artefato, Encantamento ou Terreno), depois gera as habilidades com o modelo daquele tipo e só então prevê custo e poder/resistência. Instants/sorceries não têm habilidades ativadas nem P/T; terrenos não têm custo de mana nem P/T.

Com a opção **Automático (pelo nome)**, o tipo é inferido a partir do nome usando POS tagging + WordNet (ver `src/pipeline/tipo_por_nome.py`): verbos/ações puxam instantânea/feitiço, lugares puxam terreno, profissões/nomes próprios puxam criatura, objetos puxam artefato e conceitos abstratos puxam encantamento. Se houver empate ou nenhuma pista, o tipo é sorteado.


In [ ]:
from pipeline import (
    gerar_carta,
    COLOR_PIE,
    PESO_COR_PADRAO,
    TIPO_ALEATORIO,
    TIPO_AUTOMATICO,
    TIPOS_DISPONIVEIS,
)

print("Pipeline carregado. Cores disponiveis:", list(COLOR_PIE))
print("Tipos disponiveis:", list(TIPOS_DISPONIVEIS))
print("Peso padrao da cor na geracao (0=caos total, 1=so vocabulario da cor):", PESO_COR_PADRAO)


### Teste rápido (sem usar os campos abaixo)

A geração agora é uma cadeia de Markov (bigramas/trigramas) treinada por **tipo e cor**, com peso ajustável entre o vocabulário típico da cor e um modelo geral -- **sem** filtro de color pie, de propósito (ver decisão de design em `src/pipeline/generator.py`). Rodar a mesma carta várias vezes deve dar textos diferentes.

O tipo pode ser escolhido, inferido pelo nome (`Automático (pelo nome)`) ou sorteado (`Aleatório`). No modo automático, nomes como "Serra Angel" (profissão), "Forest" (lugar), "Sword" (objeto) e "Destroy" (verbo) já sugerem o tipo; nomes sem pista caem no sorteio.


In [ ]:
exemplos = [
    ("Murder", "Preto", 0.7, TIPO_AUTOMATICO),             # verbo/ação -> instantânea/feitiço
    ("Serra Angel", "Branco", 0.7, TIPO_AUTOMATICO),       # profissão -> criatura
    ("Floresta Ancestral", "Verde", 0.7, TIPO_AUTOMATICO), # lugar -> terreno
    ("Mind Stone", "Azul", 0.7, TIPO_AUTOMATICO),          # objeto -> artefato
    ("Pacifism", "Branco", 0.7, TIPO_AUTOMATICO),          # conceito abstrato -> encantamento
    ("Lightning Bolt", "Vermelho", 0.7, TIPO_AUTOMATICO),  # fenômeno -> instantânea
    ("jasdhahs12123", "Verde", 0.7, TIPO_ALEATORIO),       # sem pista -> tipo sorteado
]

for nome_exemplo, cor_exemplo, peso_exemplo, tipo_exemplo in exemplos:
    c = gerar_carta(
        nome_exemplo, cor_exemplo, peso_cor=peso_exemplo, tipo=tipo_exemplo
    )
    print(f"{nome_exemplo!r} + {cor_exemplo} + {tipo_exemplo} (peso_cor={peso_exemplo}):")
    print(f"  tipo: {c.tipo_linha} | custo: {c.custo_mana or '(nenhum -- terreno)'}")
    print(f"  como o tipo foi decidido: {c.tipo_origem} -- {c.tipo_motivo}")
    if c.poder is not None:
        print(f"  P/T: {c.poder}/{c.resistencia}")
    print(f"  semente: {c.semente_usada or '(nenhuma)'}")
    print(f"  texto: {c.texto_regras}\n")


## 3. Gerar sua carta

Preencha o nome, escolha a cor e o tipo, ajuste o quanto a cor deve pesar na geração, depois clique em **Gerar carta**.

- **Automático (pelo nome)** (padrão): tenta inferir o tipo pelas palavras do nome; se não houver pista ou houver empate, sorteia.
- **Aleatório**: sorteia direto, sem inferência (criaturas são mais comuns, como no corpus real).
- Os demais itens forçam um tipo específico.

Cada tipo recebe subtipos sorteados a partir das probabilidades do corpus (ex.: **Creature — Human Warrior**, **Artifact — Equipment**, **Land — Forest Island**). Criaturas têm 1–3 subtipos, artefatos 0–1, terrenos 0–2, encantamentos 0–2 e mágicas 0–1.

Auras sempre têm uma linha **Enchant ...**, Equipamentos sempre têm **Equip ...**, e terrenos com tipo básico ganham a habilidade de mana correspondente (ex.: **{T}: Add {G}** para Forest).

Terrenos são gerados sem custo de mana e sem P/T; instants e feitiços não têm habilidades ativadas nem P/T.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

campo_nome = widgets.Text(
    description="Nome:",
    placeholder="ex: Murder, Serra Angel, Forest, Sword...",
    style={"description_width": "initial"},
)
campo_cor = widgets.Dropdown(
    options=list(COLOR_PIE.keys()),
    description="Cor:",
    style={"description_width": "initial"},
)
campo_tipo = widgets.Dropdown(
    options=list(TIPOS_DISPONIVEIS),
    value=TIPO_AUTOMATICO,
    description="Tipo:",
    style={"description_width": "initial"},
)
campo_peso_cor = widgets.FloatSlider(
    value=PESO_COR_PADRAO, min=0.0, max=1.0, step=0.1,
    description="Peso da cor:",
    style={"description_width": "initial"},
    readout_format=".1f",
)
botao_gerar = widgets.Button(description="Gerar carta", button_style="success")
saida = widgets.Output()


def ao_clicar(_botao):
    with saida:
        clear_output(wait=True)
        carta = gerar_carta(
            campo_nome.value, campo_cor.value,
            peso_cor=campo_peso_cor.value, tipo=campo_tipo.value,
        )
        display(carta.imagem_final)
        print(f"Nome: {carta.nome}")
        print(f"Tipo: {carta.tipo_linha}")
        print(f"Como o tipo foi decidido: {carta.tipo_origem} -- {carta.tipo_motivo}")
        print(f"Custo de mana: {carta.custo_mana or '(nenhum -- terreno)'}")
        if carta.poder is not None:
            print(f"Poder/Resistência: {carta.poder}/{carta.resistencia}")
        print(f"Texto: {carta.texto_regras}")
        print()
        print(f"[debug] peso da cor usado: {carta.peso_cor}")
        print(f"[debug] palavra-semente: {carta.semente_usada or '(nenhuma -- cadeia comecou do zero)'}")


botao_gerar.on_click(ao_clicar)
display(campo_nome, campo_cor, campo_tipo, campo_peso_cor, botao_gerar, saida)
